# 01 · Data preparation: sampling, translation, rater sheets

Builds every input the pre-registered analysis needs **without running any model on the evaluation set**:

1. Load MedMCQA, repair the known "rt"-deletion artifact, apply pre-registered filters, draw the stratified evaluation set (n = 500) and a disjoint DPO training pool (n = 500).
2. Fix the three option orderings per item (H3).
3. Timing run on 20 **training-pool** items (never evaluation items).
4. Translate EN → HA (evaluation + training pool) and back-translate HA → EN (evaluation only, for H2).
5. Export human-validation sheets for two raters and a data manifest with SHA-256 hashes to commit alongside the frozen pre-registration.

Everything is cached on Google Drive, so after a disconnect just re-run from the top.

## 0 · Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/avvas200/hausa-med-qa.git"   # edit if different
REPO_DIR = "/content/hausa-med-qa"
import os
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Optional now, required later for gated models (Llama 3.2, MedGemma).
# Store your token in Colab's Secrets panel as HF_TOKEN; never paste it into the notebook.
from google.colab import userdata
from huggingface_hub import login
try:
    login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
    print("Logged in to Hugging Face")
except Exception as e:
    print("No HF_TOKEN secret found (fine for this notebook):", e)

In [ ]:
import json, time, random, platform
import pandas as pd
import torch, transformers, datasets
from src import config as C, data as D
C.ensure_dirs()
print("Artefacts ->", C.ROOT)
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)

## 1 · License check
Record the dataset license **before** using the data. Read the dataset card yourself as well; the metadata field can be missing or incomplete.

In [ ]:
from huggingface_hub import dataset_info
info = dataset_info(C.MEDMCQA)
license_field = getattr(info.card_data, "license", None)
print("MedMCQA license (card metadata):", license_field)
print("Card: https://huggingface.co/datasets/" + C.MEDMCQA)

## 2 · Load, repair and filter (pre-registered rules in `src/config.py`)

In [ ]:
val_raw = D.load_split(C.EVAL_SPLIT)
train_raw = D.load_split(C.TRAIN_SPLIT)
val_rep, val_repairs, val_changed = D.repair_df(val_raw)
train_rep, train_repairs, _ = D.repair_df(train_raw)
print("validation rows changed by repair:", int(val_changed.sum()), "| replacements:", sum(val_repairs.values()))
print("train replacements:", sum(train_repairs.values()))
print("\nTop repairs (validation):")
for p, n in sorted(val_repairs.items(), key=lambda x: -x[1])[:10]:
    print(f"  {n:5d}  {p}")

In [ ]:
# Check a sample of repaired questions: every change should be a correct fix.
idx = [i for i, c in enumerate(val_changed) if c]
for i in random.Random(0).sample(idx, min(8, len(idx))):
    print("BEFORE:", val_raw.loc[i, "question"][:160])
    print("AFTER :", val_rep.loc[i, "question"][:160], "\n")

In [ ]:
val, val_log = D.apply_filters(val_rep)
train, train_log = D.apply_filters(train_rep)
print("validation:", val_log)
print("train     :", train_log)

In [ ]:
# Eyeball what the position-dependent / image filters actually remove.
import re
opt_pat = re.compile("|".join(C.EXCLUDE_OPTION_PATTERNS), re.I)
removed = val_rep[val_rep[D.OPTS].apply(lambda r: any(opt_pat.search(str(x)) for x in r), axis=1)]
removed[["question"] + D.OPTS].sample(8, random_state=0)

## 3 · Evaluation set (n = 500, stratified by subject)

In [ ]:
eval_df = D.stratified_sample(val, C.N_EVAL, C.SEED)
eval_recs = D.to_records(eval_df, C.SEED)
D.write_jsonl(eval_recs, C.DATA / "eval_en.jsonl")

print(eval_df["subject_name"].value_counts().to_string())
print("\nGold-answer position balance:", eval_df["cop"].value_counts().sort_index().to_dict())

## 4 · DPO training pool (n = 500, disjoint from evaluation)

In [ ]:
train_cand = D.stratified_sample(train, int(C.N_TRAIN_POOL * C.TRAIN_OVERSAMPLE), C.SEED)
train_cand, n_dropped = D.drop_overlaps(train_cand, eval_df)
print("dropped for overlap with eval:", n_dropped)
assert len(train_cand) >= C.N_TRAIN_POOL, "Too many overlaps: raise TRAIN_OVERSAMPLE and log it as a deviation"
pool_df = train_cand.head(C.N_TRAIN_POOL).reset_index(drop=True)
pool_recs = D.to_records(pool_df, C.SEED + 1)
D.write_jsonl(pool_recs, C.DATA / "train_pool_en.jsonl")
print("training pool:", len(pool_recs))

## 5 · Timing run (20 training-pool items only)

In [ ]:
from collections import Counter
from src.translate import Translator, translate_segments, is_degenerate
tr = Translator(C.NLLB_MODEL)
T = C.TRANSLATION_TAG
print("Translator:", C.NLLB_MODEL, "| tag:", T)

timing_segs = D.segments(pool_recs[:20])
t0 = time.time()
timing_out, timing_status = translate_segments(tr, timing_segs, C.CACHE / f"timing_en2ha_{T}.jsonl", C.EN, C.HA)
dt = time.time() - t0
print("statuses:", Counter(timing_status.values()))
per_seg = dt / len(timing_segs)
total_segs = 2 * 5 * C.N_EVAL + 5 * C.N_TRAIN_POOL   # eval there-and-back + pool forward
print(f"{len(timing_segs)} segments in {dt:.1f}s -> ~{per_seg*total_segs/60:.0f} min for all translation")

In [ ]:
# Spot-check the timing output before committing hours to the full run.
for k in list(timing_segs)[:15]:
    print(f"[{timing_status[k]}] EN: {timing_segs[k]}\n{' ' * (len(timing_status[k]) + 3)}HA: {timing_out[k]}\n")

## 6 · Full translation (resumable; re-run the cell after a disconnect)

In [ ]:
eval_segs = D.segments(eval_recs)
pool_segs = D.segments(pool_recs)

print("Evaluation EN -> HA")
eval_ha_map, eval_ha_status = translate_segments(tr, eval_segs, C.CACHE / f"eval_en2ha_{T}.jsonl", C.EN, C.HA)
print("Training pool EN -> HA")
pool_ha_map, pool_ha_status = translate_segments(tr, pool_segs, C.CACHE / f"pool_en2ha_{T}.jsonl", C.EN, C.HA)

# Back-translation: segments kept in English (passthrough / fallback_en) are
# copied back verbatim, never fed to NLLB as if they were Hausa.
keep_en = [k for k, s in eval_ha_status.items() if s in ("passthrough", "fallback_en")]
back_in = {k: (eval_segs[k] if k in keep_en else v) for k, v in eval_ha_map.items()}
print("Evaluation HA -> EN (translate-test, H2)")
eval_ha2en_map, eval_ha2en_status = translate_segments(
    tr, back_in, C.CACHE / f"eval_ha2en_{T}.jsonl", C.HA, C.EN, copy_keys=keep_en)
print("Inputs truncated at max_length:", tr.truncated)

In [ ]:
eval_ha = D.apply_translations(eval_recs, eval_ha_map, "ha")
eval_ha2en = D.apply_translations(eval_recs, eval_ha2en_map, "ha2en")
pool_ha = D.apply_translations(pool_recs, pool_ha_map, "ha")
D.write_jsonl(eval_ha, C.DATA / "eval_ha.jsonl")
D.write_jsonl(eval_ha2en, C.DATA / "eval_ha2en.jsonl")
D.write_jsonl(pool_ha, C.DATA / "train_pool_ha.jsonl")

## 7 · Automatic sanity checks
High "unchanged" rates are expected for drug names and numbers, but not for full question stems. Extreme length ratios usually mean hallucinated or truncated output.

In [ ]:
rows = []
for k, src in eval_segs.items():
    hyp = eval_ha_map[k]
    rows.append({"key": k, "field": "question" if k.endswith("|q") else "option",
                 "status": eval_ha_status[k],
                 "degenerate": is_degenerate(src, hyp),
                 "unchanged": D.normalize(hyp) == D.normalize(src),
                 "len_ratio": len(hyp) / max(1, len(src))})
chk = pd.DataFrame(rows)
print("Status counts (EN -> HA, evaluation):")
print(pd.crosstab(chk.field, chk.status, margins=True))
print("\nStill degenerate after safeguards:", int(chk.degenerate.sum()))
print(chk.groupby("field")[["unchanged"]].mean().round(3))
print(chk.groupby("field")["len_ratio"].describe().round(2))

# Items with any English fallback segment (reported; sensitivity analysis excludes them)
fallback_items = sorted({k.split("|")[0] for k, s in eval_ha_status.items() if s == "fallback_en"})
print("\nItems with >=1 fallback_en segment:", len(fallback_items), "of", len(eval_recs))
json.dump(fallback_items, open(C.DATA / "fallback_item_ids.json", "w"))

In [ ]:
# Read these yourself: retried and remaining outlier segments.
for k in chk[chk.status == "retried"].key.head(10):
    print(f"[retried] EN: {eval_segs[k]}\n          HA: {eval_ha_map[k]}\n")
outliers = chk[((chk.len_ratio > 3) | (chk.len_ratio < 0.3)) & (chk.status == "translated")]
print("Length-ratio outliers among translated segments:", len(outliers))
for k in outliers.key.head(10):
    print(f"  EN: {eval_segs[k]}\n  HA: {eval_ha_map[k]}\n")

## 8 · Human-validation sheets (75 items, two raters)
Each rater fills their own copy independently. Scale for `adequacy`: 1 = meaning lost, 2 = major errors, 3 = partly preserved, 4 = minor errors, 5 = fully preserved. `term_error` = 1 if a medical term was mistranslated; an English loanword kept in Hausa is **not** an error.

In [ ]:
rng = random.Random(C.SEED + 2)
val_ids = rng.sample([r["id"] for r in eval_recs], C.N_HUMAN_VALIDATION)
by_id_en = {r["id"]: r for r in eval_recs}
by_id_ha = {r["id"]: r for r in eval_ha}
sheet = []
for i in val_ids:
    en, ha = by_id_en[i], by_id_ha[i]
    row = {"id": i, "subject": en["subject"], "question_en": en["question"], "question_ha": ha["question"]}
    for j, L in enumerate(C.LETTERS):
        row[f"option_{L}_en"] = en["options"][j]
        row[f"option_{L}_ha"] = ha["options"][j]
    row.update({"adequacy_1to5": "", "term_error_0or1": "", "notes": ""})
    sheet.append(row)
sheet = pd.DataFrame(sheet)
for rater in ("rater1", "rater2"):
    sheet.to_csv(C.DATA / f"validation_{rater}.csv", index=False, encoding="utf-8-sig")
json.dump(val_ids, open(C.DATA / "validation_ids.json", "w"))
print("Sheets written to", C.DATA, "- open them in Google Sheets or Excel")

## 9 · Manifest (commit `results/data_manifest.json` with the frozen pre-registration)

In [ ]:
files = ["eval_en.jsonl", "eval_ha.jsonl", "eval_ha2en.jsonl",
         "train_pool_en.jsonl", "train_pool_ha.jsonl", "validation_ids.json",
         "fallback_item_ids.json"]
manifest = {
    "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    "seed": C.SEED,
    "dataset": C.MEDMCQA,
    "dataset_license_field": license_field,
    "translator": C.NLLB_MODEL,
    "translation_settings": {"main": C.GEN_MAIN, "retry": C.GEN_RETRY,
                             "max_new": [C.MAX_NEW_FACTOR, C.MAX_NEW_BIAS]},
    "excluded_subjects": C.EXCLUDE_SUBJECTS,
    "repairs": {"validation": sum(val_repairs.values()), "train": sum(train_repairs.values()),
                "validation_rows_changed": int(val_changed.sum())},
    "translation_status": {
        "eval_en2ha": dict(Counter(eval_ha_status.values())),
        "pool_en2ha": dict(Counter(pool_ha_status.values())),
        "eval_ha2en": dict(Counter(eval_ha2en_status.values()))},
    "items_with_fallback": len(fallback_items),
    "filter_log": {"validation": val_log, "train": train_log},
    "train_pool_overlap_dropped": n_dropped,
    "translation_truncated_inputs": tr.truncated,
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": transformers.__version__, "datasets": datasets.__version__},
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "sha256": {f: D.sha256(C.DATA / f) for f in files},
}
os.makedirs("results", exist_ok=True)
json.dump(manifest, open("results/data_manifest.json", "w"), indent=2)
json.dump(manifest, open(C.RESULTS / "data_manifest.json", "w"), indent=2)
print(json.dumps(manifest, indent=2))

## Next
Paste the outputs of sections 2, 3, 4, 5, 7 and 9 back into the chat. Then:
1. Send the two rater sheets out.
2. Once ratings are back and the timing numbers are in, fill the remaining blanks in `PREREGISTRATION.md`, commit it with `results/data_manifest.json`, and record the commit hash. **That commit is the freeze.**
3. Only then run notebook 02 (evaluation).